In [2]:
# ============================================================
# 0. Install extra dependencies
# ============================================================
!pip install -q pyclipper shapely

In [3]:
# ============================================================
# 1. Imports, paths, reproducibility
# ============================================================
import os, glob, re, random, unicodedata, subprocess
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

from PIL import Image, ImageDraw, ImageFont
import pyclipper
from shapely.geometry import Polygon

# ---- Dataset paths (standard Kaggle VinText location) ----
BASE      = "/kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext"
TRAIN_DIR = os.path.join(BASE, "train_images")
VAL_DIR   = os.path.join(BASE, "val_image")
TEST_DIR  = os.path.join(BASE, "test_image")
LABEL_DIR = os.path.join(BASE, "labels")

# ---- Output directory ----
OUT_DIR = "/kaggle/working/inference_results"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
# ============================================================
# 2. Vietnamese alphabet — NFD, 106 chars + CTC blank
#    Identical to both training notebooks
# ============================================================
SPECIAL_BLANK = "<BLANK>"

VN_BASE_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
    "\u0301"   # acute
    "\u0300"   # grave
    "\u0309"   # hook above
    "\u0323"   # dot below
    "\u0303"   # tilde
    "\u0302"   # circumflex
    "\u0306"   # breve
    "\u031b"   # horn
    "\u0111"   # d with stroke
    "\u0110"   # D with stroke
)

_seen, _char_set = set(), []
for c in VN_BASE_CHARS:
    if c not in _seen:
        _char_set.append(c); _seen.add(c)

CHARS       = [SPECIAL_BLANK] + _char_set
CHAR2IDX    = {c: i for i, c in enumerate(CHARS)}
IDX2CHAR    = {i: c for c, i in CHAR2IDX.items()}
NUM_CLASSES = len(CHARS)
print(f"Alphabet size (incl. CTC blank): {NUM_CLASSES}")


def indices_to_text(indices) -> str:
    return "".join(IDX2CHAR.get(i, "") for i in indices if i != 0)


def normalize_for_match(s: str) -> str:
    return unicodedata.normalize("NFD", s).strip()

Alphabet size (incl. CTC blank): 106


In [5]:
# ============================================================
# 3. VinText annotation parser
# ============================================================
def parse_annotation(gt_path: str) -> List[Tuple[np.ndarray, str]]:
    records = []
    if not os.path.exists(gt_path):
        return records
    with open(gt_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 8)
            if len(parts) < 9:
                continue
            try:
                coords = list(map(float, parts[:8]))
            except ValueError:
                continue
            transcript = parts[8].strip()
            pts = np.array(coords, dtype=np.float32).reshape(4, 2)
            records.append((pts, transcript))
    return records


def _img_idx_from_name(name: str):
    m = re.search(r"(\d+)", os.path.basename(name))
    return int(m.group(1)) if m else None

In [6]:
# ============================================================
# 4. Image preprocessing utilities
# ============================================================
DET_INPUT_SIZE = 640
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

IMG_H     = 32
IMG_W_MAX = 512


def letterbox_image(img: np.ndarray, size: int = DET_INPUT_SIZE):
    """Resize keeping aspect ratio, then pad to (size, size)."""
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=resized.dtype)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y


def to_tensor_norm(img_bgr: np.ndarray) -> torch.Tensor:
    """BGR uint8 -> ImageNet-normalised float CHW tensor (DBNet input)."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2, 0, 1)).contiguous()


def order_quad(pts: np.ndarray) -> np.ndarray:
    """Reorder 4 points as [TL, TR, BR, BL]."""
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = pts[:, 1] - pts[:, 0]
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def crop_quad(image_bgr: np.ndarray, pts: np.ndarray, target_h: int = IMG_H) -> np.ndarray:
    """Perspective-warp a 4-point quadrilateral to a target_h × W strip."""
    pts = pts.astype(np.float32)
    w1 = np.linalg.norm(pts[1] - pts[0]); w2 = np.linalg.norm(pts[2] - pts[3])
    h1 = np.linalg.norm(pts[3] - pts[0]); h2 = np.linalg.norm(pts[2] - pts[1])
    W = max(int((w1 + w2) / 2), 1); H = max(int((h1 + h2) / 2), 1)
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst)
    crop = cv2.warpPerspective(image_bgr, M, (W, H))
    scale = target_h / max(crop.shape[0], 1)
    new_w = max(int(crop.shape[1] * scale), 1)
    return cv2.resize(crop, (new_w, target_h))


def preprocess_crop(crop_bgr: np.ndarray) -> torch.Tensor:
    """BGR crop -> [-1, 1] CHW tensor (CRNN input)."""
    img = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.from_numpy(img.transpose(2, 0, 1))

In [7]:
# ============================================================
# 5. Model definitions — DBNet (ResNet-18 + FPN) and CRNN
# ============================================================
class FPN(nn.Module):
    def __init__(self, in_channels=(64, 128, 256, 512), inner_channels: int = 256):
        super().__init__()
        c2, c3, c4, c5 = in_channels
        self.lat5 = nn.Conv2d(c5, inner_channels, 1)
        self.lat4 = nn.Conv2d(c4, inner_channels, 1)
        self.lat3 = nn.Conv2d(c3, inner_channels, 1)
        self.lat2 = nn.Conv2d(c2, inner_channels, 1)
        self.smooth5 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth4 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth3 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth2 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)

    def forward(self, c2, c3, c4, c5):
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="bilinear", align_corners=False)
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        p5 = self.smooth5(p5); p4 = self.smooth4(p4)
        p3 = self.smooth3(p3); p2 = self.smooth2(p2)
        size = p2.shape[-2:]
        p3 = F.interpolate(p3, size=size, mode="bilinear", align_corners=False)
        p4 = F.interpolate(p4, size=size, mode="bilinear", align_corners=False)
        p5 = F.interpolate(p5, size=size, mode="bilinear", align_corners=False)
        return torch.cat([p2, p3, p4, p5], dim=1)


class DBNet(nn.Module):
    def __init__(self, k: int = 50, pretrained: bool = False):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.fpn    = FPN()
        inner = 256

        def _head():
            return nn.Sequential(
                nn.Conv2d(inner, inner // 4, 3, padding=1, bias=False),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, inner // 4, 2, stride=2),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, 1, 2, stride=2),
            )

        self.prob_head   = _head()
        self.thresh_head = _head()
        self.k = k

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        feat   = self.fpn(c2, c3, c4, c5)
        prob   = torch.sigmoid(self.prob_head(feat))
        thresh = torch.sigmoid(self.thresh_head(feat))
        binary = torch.sigmoid(self.k * (prob - thresh))
        return prob, thresh, binary


class CRNN(nn.Module):
    """CNN + BiLSTM + CTC (Shi et al., 2015)."""
    def __init__(self, num_classes: int, hidden_size: int = 256, n_rnn_layers: int = 2):
        super().__init__()

        def _block(ci, co, pool=None):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                      nn.LeakyReLU(0.2, inplace=True)]
            if pool == "full":
                layers.append(nn.MaxPool2d(2, 2))
            elif pool == "height":
                layers.append(nn.MaxPool2d((2, 1), (2, 1)))
            return nn.Sequential(*layers)

        self.cnn = nn.Sequential(
            _block(3,   64,  "full"),
            _block(64,  128, "full"),
            _block(128, 256),
            _block(256, 256, "height"),
            _block(256, 512),
            _block(512, 512, "height"),
            _block(512, 512, "height"),
            nn.Conv2d(512, 512, (1, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn = nn.LSTM(512, hidden_size, num_layers=n_rnn_layers,
                           batch_first=False, bidirectional=True,
                           dropout=0.2 if n_rnn_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        b, c, h, w = feat.size()
        if h > 1:
            feat = F.adaptive_avg_pool2d(feat, (1, w))
        feat = feat.squeeze(2).permute(2, 0, 1)
        out, _ = self.rnn(feat)
        return F.log_softmax(self.fc(out), dim=2)


def ctc_greedy_decode(log_probs: torch.Tensor) -> List[str]:
    _, idx = log_probs.max(2)
    out = []
    for seq in idx.permute(1, 0):
        seq = seq.tolist(); prev = None; chars = []
        for i in seq:
            if i != prev and i != 0:
                chars.append(i)
            prev = i
        out.append(indices_to_text(chars))
    return out

In [8]:
# ============================================================
# 6. DBNet post-processing: probability map -> quad polygons
# ============================================================
@torch.no_grad()
def decode_db_polygons(prob_map: torch.Tensor,
                       prob_thresh: float = 0.3,
                       box_thresh: float = 0.55,
                       min_size: float = 3.0,
                       max_candidates: int = 1000,
                       unclip_ratio: float = 1.5):
    results = []
    prob_np = prob_map.detach().squeeze(1).cpu().numpy()
    binary  = (prob_np >= prob_thresh).astype(np.uint8)
    for prob, b in zip(prob_np, binary):
        contours, _ = cv2.findContours(b, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        polys = []
        for cnt in contours[:max_candidates]:
            if cnt.shape[0] < 4:
                continue
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            approx  = cv2.approxPolyDP(cnt, epsilon, True).reshape(-1, 2)
            if approx.shape[0] < 4:
                rect = cv2.minAreaRect(cnt)
                approx = cv2.boxPoints(rect).astype(np.float32)
            mask = np.zeros_like(prob, dtype=np.uint8)
            cv2.fillPoly(mask, [approx.astype(np.int32)], 1)
            if mask.sum() == 0:
                continue
            score = float(prob[mask == 1].mean())
            if score < box_thresh:
                continue
            try:
                p = Polygon(approx.astype(np.float64))
                if (not p.is_valid) or p.area < min_size:
                    continue
                distance = p.area * unclip_ratio / max(p.length, 1e-6)
            except Exception:
                continue
            pco = pyclipper.PyclipperOffset()
            try:
                pco.AddPath(np.round(approx).astype(int).tolist(),
                            pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
                expanded = pco.Execute(distance)
            except Exception:
                continue
            if not expanded:
                continue
            poly = np.asarray(expanded[0], dtype=np.float32)
            if poly.shape[0] < 4:
                continue
            rect = cv2.minAreaRect(poly)
            quad = cv2.boxPoints(rect).astype(np.float32)
            quad = order_quad(quad)
            polys.append((quad, score))
        results.append(polys)
    return results

In [9]:
# ============================================================
# 7. Inference pipelines
# ============================================================
class DetectionPipeline:
    """DBNet-only: returns predicted quads in original image coords."""
    def __init__(self, detector: DBNet, device: torch.device,
                 det_size: int = DET_INPUT_SIZE,
                 prob_thresh: float = 0.3,
                 box_thresh: float = 0.55,
                 unclip_ratio: float = 1.5):
        self.detector     = detector.to(device).eval()
        self.device       = device
        self.det_size     = det_size
        self.prob_thresh  = prob_thresh
        self.box_thresh   = box_thresh
        self.unclip_ratio = unclip_ratio

    @torch.no_grad()
    def __call__(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, float]]:
        h0, w0 = image_bgr.shape[:2]
        canvas, scale, px, py = letterbox_image(image_bgr, self.det_size)
        tensor = to_tensor_norm(canvas).unsqueeze(0).to(self.device)
        prob, _, _ = self.detector(tensor)
        decoded = decode_db_polygons(
            prob, self.prob_thresh, self.box_thresh,
            unclip_ratio=self.unclip_ratio,
        )[0]
        out = []
        for quad, score in decoded:
            q = quad.copy()
            q[:, 0] = np.clip((q[:, 0] - px) / max(scale, 1e-6), 0, w0 - 1)
            q[:, 1] = np.clip((q[:, 1] - py) / max(scale, 1e-6), 0, h0 - 1)
            out.append((q.astype(np.float32), score))
        return out


class SpotterPipeline:
    """DBNet + CRNN end-to-end: returns (quad, text, score) tuples."""
    def __init__(self, detector: DBNet, recognizer: CRNN, device: torch.device,
                 det_size: int = DET_INPUT_SIZE,
                 prob_thresh: float = 0.3,
                 box_thresh: float = 0.55,
                 unclip_ratio: float = 1.5,
                 min_quad_size: int = 4):
        self.detector      = detector.to(device).eval()
        self.recognizer    = recognizer.to(device).eval()
        self.device        = device
        self.det_size      = det_size
        self.prob_thresh   = prob_thresh
        self.box_thresh    = box_thresh
        self.unclip_ratio  = unclip_ratio
        self.min_quad_size = min_quad_size

    @torch.no_grad()
    def detect(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, float]]:
        h0, w0 = image_bgr.shape[:2]
        canvas, scale, px, py = letterbox_image(image_bgr, self.det_size)
        tensor = to_tensor_norm(canvas).unsqueeze(0).to(self.device)
        prob, _, _ = self.detector(tensor)
        decoded = decode_db_polygons(
            prob, self.prob_thresh, self.box_thresh,
            unclip_ratio=self.unclip_ratio,
        )[0]
        out = []
        for quad, score in decoded:
            q = quad.copy()
            q[:, 0] = np.clip((q[:, 0] - px) / max(scale, 1e-6), 0, w0 - 1)
            q[:, 1] = np.clip((q[:, 1] - py) / max(scale, 1e-6), 0, h0 - 1)
            out.append((q.astype(np.float32), score))
        return out

    @torch.no_grad()
    def __call__(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, str, float]]:
        det_results = self.detect(image_bgr)
        if not det_results:
            return []
        spotted = []
        for quad, score in det_results:
            try:
                crop = crop_quad(image_bgr, quad.astype(np.float32))
            except Exception:
                continue
            if crop.shape[1] < self.min_quad_size:
                continue
            t = preprocess_crop(crop).unsqueeze(0).to(self.device)
            if t.shape[3] > IMG_W_MAX:
                t = t[:, :, :, :IMG_W_MAX]
            log_probs = self.recognizer(t)
            text = ctc_greedy_decode(log_probs)[0]
            spotted.append((quad, text, score))
        return spotted

In [11]:
# ============================================================
# 8. Auto-locate pretrained .pth files
#    Searches /kaggle/input recursively for the two model files.
# ============================================================
def _find_pth(filename: str) -> Optional[str]:
    """Search /kaggle/input (and /kaggle/working) for a .pth by exact filename."""
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        for root, _, files in os.walk(search_root):
            if filename in files:
                return os.path.join(root, filename)
    return None


E2E_PTH_NAME  = "dbnet_crnn_e2e_vintext_12epochs.pth"
DET_PTH_NAME  = "dbnet_resnet18_det_vintext_12epochs.pth"

E2E_PTH = _find_pth(E2E_PTH_NAME)
DET_PTH = _find_pth(DET_PTH_NAME)

print(f"E2E model  : {E2E_PTH  or '*** NOT FOUND — upload ' + E2E_PTH_NAME + ' as a Kaggle dataset input ***'}")
print(f"Det model  : {DET_PTH  or '*** NOT FOUND — upload ' + DET_PTH_NAME + ' as a Kaggle dataset input ***'}")

E2E model  : /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/dbnet_crnn_e2e_vintext_12epochs.pth
Det model  : /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth


In [12]:
# ============================================================
# 9. Load models
# ============================================================
def _load_ckpt(path: str):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


# ---- Detection-only model ----
det_only_model = None
det_pipeline   = None
if DET_PTH:
    print(f"Loading DBNet (det-only) from {DET_PTH} ...")
    ckpt_det = _load_ckpt(DET_PTH)
    det_only_model = DBNet(pretrained=False).to(DEVICE)
    # The det-only checkpoint may store state under 'detector_state' key
    state_det = ckpt_det.get("detector_state", ckpt_det)
    det_only_model.load_state_dict(state_det, strict=True)
    det_only_model.eval()
    det_pipeline = DetectionPipeline(det_only_model, DEVICE)
    print("  DBNet (det-only) loaded successfully.")
else:
    print("[SKIP] DBNet det-only model not found.")


# ---- E2E model (DBNet + CRNN) ----
e2e_pipeline = None
if E2E_PTH:
    print(f"\nLoading DBNet+CRNN E2E from {E2E_PTH} ...")
    ckpt_e2e = _load_ckpt(E2E_PTH)
    e2e_detector   = DBNet(pretrained=False).to(DEVICE)
    e2e_recognizer = CRNN(NUM_CLASSES).to(DEVICE)
    e2e_detector.load_state_dict(ckpt_e2e["detector_state"], strict=True)
    e2e_recognizer.load_state_dict(ckpt_e2e["recognizer_state"], strict=True)
    e2e_detector.eval()
    e2e_recognizer.eval()
    e2e_pipeline = SpotterPipeline(e2e_detector, e2e_recognizer, DEVICE)
    print("  DBNet+CRNN E2E loaded successfully.")
else:
    print("[SKIP] E2E model not found.")

Loading DBNet (det-only) from /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth ...
  DBNet (det-only) loaded successfully.

Loading DBNet+CRNN E2E from /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/dbnet_crnn_e2e_vintext_12epochs.pth ...
  DBNet+CRNN E2E loaded successfully.


In [13]:
# ============================================================
# 10. Download NotoSans font for Vietnamese text rendering
# ============================================================
FONT_PATH = "/kaggle/working/NotoSans-Regular.ttf"
FONT_URL  = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf"

if not os.path.exists(FONT_PATH):
    try:
        subprocess.run(["wget", "-q", "-O", FONT_PATH, FONT_URL], check=True)
        print(f"Font downloaded -> {FONT_PATH}")
    except Exception as e:
        print(f"Font download failed ({e}); will fall back to PIL default font.")
else:
    print(f"Font already present: {FONT_PATH}")


def _font(size: int) -> ImageFont.FreeTypeFont:
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def draw_text_pil(canvas_bgr: np.ndarray, text: str, position: Tuple[int, int],
                  color_rgb: Tuple[int, int, int],
                  font_size: int = 18,
                  bg_rgb: Tuple[int, int, int] = (0, 0, 0)) -> np.ndarray:
    """Render `text` onto a BGR numpy image using PIL/NotoSans, with background box."""
    pil  = Image.fromarray(cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    font = _font(font_size)
    bbox = draw.textbbox(position, text, font=font)
    pad  = 2
    draw.rectangle([bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad], fill=bg_rgb)
    draw.text(position, text, font=font, fill=color_rgb)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

Font downloaded -> /kaggle/working/NotoSans-Regular.ttf


In [14]:
# ============================================================
# 11. Core visualisation helper
#     Left  : original image + GT boxes (green) + GT text labels
#     Right : predicted detection/E2E boxes (cyan) + recognised text
# ============================================================

# Colours (BGR for OpenCV polylines, RGB for PIL text)
GT_BOX_BGR   = (0,   200,   0)   # green
GT_TEXT_RGB  = (0,   230,   0)   # green
GT_TEXTBG    = (0,    40,   0)

PRED_BOX_BGR  = (255, 200,   0)  # cyan-ish (matches e2e notebook)
PRED_TEXT_RGB = (0,   215, 255)  # cyan
PRED_TEXTBG   = (20,   20,  80)


def _annotate_gt(img_bgr: np.ndarray,
                 gt: List[Tuple[np.ndarray, str]],
                 font_size: int) -> np.ndarray:
    """Draw GT boxes + labels on a copy of the image."""
    canvas = img_bgr.copy()
    for poly, txt in gt:
        cv2.polylines(canvas, [poly.astype(np.int32)], True, GT_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 4)
        canvas = draw_text_pil(
            canvas, f"GT: {txt}",
            (x, label_y),
            color_rgb=GT_TEXT_RGB,
            font_size=font_size,
            bg_rgb=GT_TEXTBG,
        )
    return canvas


def _annotate_det_pred(img_bgr: np.ndarray,
                       preds: List[Tuple[np.ndarray, float]],
                       font_size: int) -> np.ndarray:
    """Draw detection-only predicted boxes (no text label) on a copy of the image."""
    canvas = img_bgr.copy()
    for poly, score in preds:
        cv2.polylines(canvas, [poly.astype(np.int32)], True, PRED_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 2)
        canvas = draw_text_pil(
            canvas, f"{score:.2f}",
            (x, label_y),
            color_rgb=PRED_TEXT_RGB,
            font_size=font_size,
            bg_rgb=PRED_TEXTBG,
        )
    return canvas


def _annotate_e2e_pred(img_bgr: np.ndarray,
                       preds: List[Tuple[np.ndarray, str, float]],
                       font_size: int) -> np.ndarray:
    """Draw E2E predicted boxes + recognised text on a copy of the image."""
    canvas = img_bgr.copy()
    for poly, text, score in preds:
        cv2.polylines(canvas, [poly.astype(np.int32)], True, PRED_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 2)
        canvas = draw_text_pil(
            canvas, f"PR: {text} ({score:.2f})",
            (x, label_y),
            color_rgb=PRED_TEXT_RGB,
            font_size=font_size,
            bg_rgb=PRED_TEXTBG,
        )
    return canvas


def save_side_by_side(img_bgr: np.ndarray,
                      right_canvas: np.ndarray,
                      title_left: str,
                      title_right: str,
                      save_path: str,
                      suptitle: str = "") -> None:
    """Save a 1×2 matplotlib figure: left = GT-annotated, right = predicted."""
    fig, axes = plt.subplots(1, 2, figsize=(22, 11))
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(title_left,  fontsize=13, fontweight="bold")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(right_canvas, cv2.COLOR_BGR2RGB))
    axes[1].set_title(title_right, fontsize=13, fontweight="bold")
    axes[1].axis("off")
    if suptitle:
        plt.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  saved -> {save_path}")

In [15]:
# ============================================================
# 12. Run inference — configuration
# ============================================================
# Change SAMPLE_SPLIT to 'val' or 'train' if you prefer a different split.
SAMPLE_SPLIT = "test"   # 'test' | 'val' | 'train'
N_SAMPLES    = 6        # how many images to visualise per model
RANDOM_SEED_SAMPLE = 42 # set None for first-N images, integer for random selection

SPLIT_DIR = {"test": TEST_DIR, "val": VAL_DIR, "train": TRAIN_DIR}[SAMPLE_SPLIT]

all_image_paths = sorted(
    glob.glob(os.path.join(SPLIT_DIR, "*.jpg")) +
    glob.glob(os.path.join(SPLIT_DIR, "*.png"))
)
print(f"Total {SAMPLE_SPLIT} images found: {len(all_image_paths)}")

# Pick N_SAMPLES images (randomly or first-N)
if RANDOM_SEED_SAMPLE is not None:
    rng = random.Random(RANDOM_SEED_SAMPLE)
    sample_paths = rng.sample(all_image_paths, min(N_SAMPLES, len(all_image_paths)))
    sample_paths = sorted(sample_paths)     # consistent order for output filenames
else:
    sample_paths = all_image_paths[:N_SAMPLES]

print(f"Selected {len(sample_paths)} images for inference.")

Total test images found: 500
Selected 6 images for inference.


In [16]:
# ============================================================
# 13a. Detection-only inference visualisation
#      Left : original + GT boxes (green)
#      Right: same image + DBNet predicted boxes (cyan) + confidence score
# ============================================================
if det_pipeline is None:
    print("[SKIP] DBNet det-only model not loaded.")
else:
    viz_det_dir = os.path.join(OUT_DIR, "det_only_samples")
    os.makedirs(viz_det_dir, exist_ok=True)
    print(f"\n=== DBNet Detection-Only Inference ({SAMPLE_SPLIT} split) ===")

    for ip in sample_paths:
        img = cv2.imread(ip)
        if img is None:
            print(f"  [warn] Could not read {ip}")
            continue

        idx = _img_idx_from_name(ip)
        gt  = parse_annotation(os.path.join(LABEL_DIR, f"gt_{idx}.txt"))
        # Filter out ignore-tagged instances for the GT panel
        gt_visible = [(p, t) for p, t in gt if t not in ("###", "***", "")]

        font_size = max(14, img.shape[0] // 55)

        # Left panel: original image + GT boxes
        gt_canvas   = _annotate_gt(img, gt_visible, font_size)

        # Right panel: predicted detection boxes
        preds       = det_pipeline(img)
        pred_canvas = _annotate_det_pred(img, preds, font_size)

        save_path = os.path.join(viz_det_dir, f"det_{Path(ip).stem}.png")
        save_side_by_side(
            gt_canvas, pred_canvas,
            title_left  = f"GT boxes ({len(gt_visible)})  —  {os.path.basename(ip)}",
            title_right = f"DBNet predicted boxes ({len(preds)})",
            save_path   = save_path,
            suptitle    = os.path.basename(ip),
        )

    print(f"\nDetection-only visualisations saved to: {viz_det_dir}")


=== DBNet Detection-Only Inference (test split) ===
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1513.png
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1558.png
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1626.png
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1641.png
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1828.png
  saved -> /kaggle/working/inference_results/det_only_samples/det_im1880.png

Detection-only visualisations saved to: /kaggle/working/inference_results/det_only_samples


In [17]:
# ============================================================
# 13b. End-to-end (DBNet + CRNN) inference visualisation
#      Left : original + GT boxes (green) + GT text
#      Right: same image + predicted boxes (cyan) + recognised Vietnamese text
# ============================================================
if e2e_pipeline is None:
    print("[SKIP] DBNet+CRNN E2E model not loaded.")
else:
    viz_e2e_dir = os.path.join(OUT_DIR, "e2e_samples")
    os.makedirs(viz_e2e_dir, exist_ok=True)
    print(f"\n=== DBNet+CRNN E2E Inference ({SAMPLE_SPLIT} split) ===")

    for ip in sample_paths:
        img = cv2.imread(ip)
        if img is None:
            print(f"  [warn] Could not read {ip}")
            continue

        idx = _img_idx_from_name(ip)
        gt  = parse_annotation(os.path.join(LABEL_DIR, f"gt_{idx}.txt"))
        gt_visible = [(p, t) for p, t in gt if t not in ("###", "***", "")]

        font_size = max(14, img.shape[0] // 55)

        # Left panel: original + GT boxes + GT text
        gt_canvas   = _annotate_gt(img, gt_visible, font_size)

        # Right panel: E2E predictions
        preds       = e2e_pipeline(img)      # List[(quad, text, score)]
        pred_canvas = _annotate_e2e_pred(img, preds, font_size)

        save_path = os.path.join(viz_e2e_dir, f"e2e_{Path(ip).stem}.png")
        save_side_by_side(
            gt_canvas, pred_canvas,
            title_left  = f"GT — {os.path.basename(ip)}  ({len(gt_visible)} boxes, green)",
            title_right = f"E2E Predicted ({len(preds)} boxes, cyan)",
            save_path   = save_path,
            suptitle    = os.path.basename(ip),
        )

    print(f"\nE2E visualisations saved to: {viz_e2e_dir}")


=== DBNet+CRNN E2E Inference (test split) ===
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1513.png
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1558.png
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1626.png
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1641.png
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1828.png
  saved -> /kaggle/working/inference_results/e2e_samples/e2e_im1880.png

E2E visualisations saved to: /kaggle/working/inference_results/e2e_samples


In [ ]:
# ============================================================
# 14. Quick inline preview of the first saved image per model
# ============================================================
import matplotlib.image as mpimg

for label, subdir in [("Detection-only", "det_only_samples"), ("E2E DBNet+CRNN", "e2e_samples")]:
    d = os.path.join(OUT_DIR, subdir)
    pngs = sorted(glob.glob(os.path.join(d, "*.png")))
    if not pngs:
        continue
    preview = pngs[0]
    fig, ax = plt.subplots(1, 1, figsize=(20, 9))
    ax.imshow(mpimg.imread(preview))
    ax.set_title(f"{label}  —  {os.path.basename(preview)}", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Preview: {preview}")